In [1]:
import pandas as pd 
import yfinance as yf
import fundainsight.app.picker as fi
import fincli.utils.web_scraper as sc
import math
import numpy as np
import json



In [ ]:

df = pd.read_csv('workspace_materials/all_df_new.csv')
df_raw = df.copy()
df_raw


In [ ]:
df_stocks = pd.DataFrame(df_raw["stock"].unique()).rename(columns={0:"Symbol"})
df_stocks_funda_hist = fi.picker(df_stocks)

df_stocks_funda_hist.to_csv('workspace_materials/df_research_stocks_funda_hist.csv', index=False)


In [ ]:
df_unfiltered = pd.read_csv('workspace_materials/funda_insight_result_unfiltered.csv_2023-11-22_17-34.csv')

df_tikers = df_unfiltered["Symbol"].unique()
all_df_list = [] 

for tkr in df_tikers:
    single_stock_pd = yf.download(tickers=tkr, start="2021-12-31", end="2022-06-30", auto_adjust=True)
    single_stock_pd['stock'] = tkr
    all_df_list.append(single_stock_pd)  # Append DataFrame to the list

df_prices = pd.concat(all_df_list)  # Concatenate all DataFrames in the list


In [ ]:
df_prices.reset_index(inplace=True)

In [ ]:



def calculate_returns(single_stock_df:pd.DataFrame):
    return  (math.log(single_stock_df['Close']/single_stock_df['Open'])+1)

# return single_stock_df
#   return 100*(single_stock_df.iloc[-1]['Close']-single_stock_df.iloc[0]['Open'])/single_stock_df.iloc[0]['Open']
def g_mean(x):
    a = np.log(x)
    return np.exp(a.mean())


In [ ]:
df_prices['month']=df_prices['Date'].dt.month
df_prices['year']=df_prices['Date'].dt.year
df_prices

In [ ]:
df_prices['day_returns'] = df_prices.apply(calculate_returns, axis=1)


In [ ]:
df_prices.sort_values(by=['stock','year','month'], inplace=True)
df_prices

In [ ]:

stocks_avg_ln_return = df_prices.groupby(['stock', 'year', 'month'])['day_returns'].apply(g_mean).reset_index()

stocks_avg_ln_return = stocks_avg_ln_return.rename(columns={'day_returns':'avg_ln_return'})
stocks_avg_ln_return

In [ ]:
stock_a = df_prices[(df_prices['stock'] == 'A') & (df_prices['year'] == 2022) & (df_prices['month'] == 1)].groupby(['stock', 'year', 'month'])['day_returns'].apply(g_mean).reset_index()
stock_a

In [ ]:
stocks_avg_ln_return.to_csv('workspace_materials/df_research_stocks_avg_ln_return.csv', index=False)

In [ ]:
df_stocks_period_return = stocks_avg_ln_return.groupby(['stock'])['avg_ln_return'].apply(lambda x: 100*(g_mean(x)-1) ).reset_index()
df_stocks_period_return.to_csv('workspace_materials/df_research_stocks_period_return.csv', index=False)

In [ ]:
df_stocks_period_return.hist(column='avg_ln_return', bins=100)

In [ ]:
# merged_df = pd.merge(df_stocks_period_return, df_unfiltered, left_on='stock', right_on='Symbol', how='inner')
merged_df = pd.merge(df_unfiltered, df_stocks_period_return, left_on='Symbol', right_on='stock', how='inner')
merged_df.drop(columns=['stock'], inplace=True)
merged_df.to_csv('workspace_materials/df_research_stocks_period_return_merged.csv', index=False)

In [ ]:
df_filtered = pd.read_csv('workspace_materials/df_research_stocks_funda_hist.csv')
merged_df_filtered = pd.merge(df_filtered, df_stocks_period_return, left_on='Symbol', right_on='stock', how='inner')
merged_df_filtered.to_csv('workspace_materials/df_research_stocks_period_return_merged_filtered.csv', index=False)

In [ ]:
headers = {
    'user-agent': 'Premier Boutique Store premierboutique12@gmail.com Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/60.0.3112.113 Safari/537.36',
    'Accept-Encoding': 'gzip, deflate, br',
    'Connection': 'keep-alive'
}

In [ ]:
response = sc.scrape("https://www.sec.gov/files/company_tickers_exchange.json",headers)

#Convert bytes to json file and save it
json_data = json.loads(response)
json.dump(json_data, open("workspace_materials/company_tickers_exchange.json", 'w'), indent=4)



In [ ]:
json_data = json.load(open("workspace_materials/company_tickers_exchange.json", 'r'))
cik_list =  [[cik[0],cik[2],cik[3]] for cik in json_data['data'] if cik[3] == 'Nasdaq' or cik[3] == 'NYSE']

for cik in cik_list:
    if len(str(cik[0])) < 10:
        cik[0] = '0'*(10-len(str(cik[0]))) + str(cik[0])
        
df_cik = pd.DataFrame(cik_list, columns=['CIK','Symbol','Exchange'])
df_cik.to_csv('workspace_materials/df_cik.csv', index=False)

 
cik_list

In [ ]:


def get_cik(cik: str) -> str | None:
    # API endpoint for company's filing history

    url = f'https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json'

    # time.sleep(1)

    # Send a request to the API

    response = sc.scrape(url, headers)  # Send a request to the API

    if response == None:
        return None
    try:
        jsonFile = json.loads(response)
    except:
        return None
    return jsonFile

def get_cik_statements(cik):
    return get_cik(cik[0])

In [ ]:
# with ThreadPoolExecutor() as executor:
#     companies_statements_history = executor.map(get_cik_statements, cik_list)
#     # Remove null values form companies_statements_history

# # companies_statements_history = [company for company in companies_statements_history if company is not None]

companies_statements_history = [ get_cik(cik[0]) for cik in cik_list]

with open('workspace_materials/companies_statements_history.json', 'w') as outfile:
    json.dump(list(companies_statements_history), outfile, indent=4)


In [ ]:
companies_statements_history_json = json.load(open("workspace_materials/companies_statements_history.json", 'r'))

companies_statements_history_json = [company for company in companies_statements_history_json if company is not None]

json.dump(companies_statements_history_json, open("workspace_materials/companies_statements_history_filtered.json", 'w'), indent=4)

In [3]:
import json

# Load your JSON data
with open('workspace_output/appl_320193.json', 'r') as file:
    data = json.load(file)

# Assuming 'facts' and 'us-gaap' keys are consistently present
facts_us_gaap = data['facts']['us-gaap']

# Relevant fields you're interested in
relevant_fields = ['AssetsCurrent', 'CommonStockSharesIssued','StockholdersEquity',]

# Dictionary to hold your extracted data
extracted_data = {field: [] for field in relevant_fields}

for field in relevant_fields:
    if field in facts_us_gaap:
        for unit in facts_us_gaap[field]['units']:
            for item in facts_us_gaap[field]['units'][unit]:
                record = {
                    'end': item['end'],
                    'val': item['val'],
                    'fy': item['fy'],
                    'fp': item['fp']
                }
                extracted_data[field].append(record)

json.dump(extracted_data, open("workspace_output/appl_320193_extracted.json", 'w'), indent=4)

 

In [ ]:

# Inputs
years = np.array([1, 2, 3, 4, 5])  # Projection period
fcf = np.array([100, 110, 120, 130, 140])  # Free cash flow projections
base_discount_rate = 0.1  # Starting discount rate
esg_adjustment_factor = -0.01  # ESG adjustment (reduces discount rate by 1% if positive ESG performance)

# Adjusted Discount Rate incorporating ESG adjustments
adjusted_discount_rate = base_discount_rate + esg_adjustment_factor

# Discount Factors and PV of FCF
discount_factors = 1 / ((1 + adjusted_discount_rate) ** years)
pv_fcf = fcf * discount_factors

# NPV and Terminal Value
npv = np.sum(pv_fcf)
terminal_value = fcf[-1] * (1 + 0.02) / (adjusted_discount_rate - 0.02)  # Assuming a 2% terminal growth rate
pv_terminal_value = terminal_value / ((1 + adjusted_discount_rate) ** years[-1])

# Total Enterprise Value
tev = npv + pv_terminal_value

# Assuming some debt and cash values for deriving equity value
debt = 200
cash_and_equivalents = 50

# Equity Value
equity_value = tev - debt + cash_and_equivalents

npv, pv_terminal_value, tev, equity_value


In [5]:
import requests
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-large")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-large")

img_url = 'https://storage.googleapis.com/sfr-vision-language-research/BLIP/demo.jpg' 
raw_image = Image.open('C://Users//Yonatan Levin//Documents//Capture2.PNG').convert('RGB')

# conditional image captioning
text = "what is the highest number in the image?"
inputs = processor(raw_image, text, return_tensors="pt")

out = model.generate(**inputs)
print(processor.decode(out[0], skip_special_tokens=True))

# unconditional image captioning
inputs = processor(raw_image, return_tensors="pt")

out = model.generate(**inputs)
print(processor.decode(out[0], skip_special_tokens=True))


c:\ProgramData\anaconda3\Lib\site-packages\transformers\generation\utils.py:1127: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


what is the highest number in the image?
a sample of a invoice form with a blue arrow pointing to the right
